In [ ]:
import os
import math
import copy
import time
from tqdm import tqdm
from dataclasses import dataclass
from typing import Union, Optional, List, Dict, Any


import GPUtil
import plotly
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import LambdaLR
from torch.nn.utils.rnn import pad_sequence

import torch.distributed as dist
import torch.multiprocessing as mp
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP

import datasets
from transformers import AutoTokenizer, PreTrainedTokenizerBase

import import_ipynb; from at251 import create_causal_mask

# import warnings
# warnings.filterwarnings("ignore")

# Part 2: Preparation for Training

This section describes the training regime for our models.


> We stop for a quick interlude to introduce some of the tools
> needed to train a standard encoder decoder model. First we define a
> batch object that holds the src and target sentences for training,
> as well as constructing the masks.

### Batching and Masking

create_causal_mask(4)

In [38]:
# 简单的 batch 格式
# simple batching format

# version 2022
class Batch:
    """Object for holding a batch of data with mask during training."""

    def __init__(self, src, tgt=None, pad=2):  # 2 = <blank>
        self.src = src
        self.src_mask = (src != pad).unsqueeze(-2)  # padding mask (unsqueeze for broadcasting)
        if tgt is not None:
            self.tgt = tgt[:, :-1]
            self.tgt_y = tgt[:, 1:]     # next token/word prediction
            self.tgt_mask = self.make_std_mask(self.tgt, pad)
            self.num_tokens = (self.tgt_y != pad).data.sum()

    @staticmethod
    def make_std_mask(tgt, pad):
        "Create a mask to hide padding and future words."
        tgt_mask = (tgt != pad).unsqueeze(-2)       # padding mask (unsqueeze for broadcasting)
        tgt_mask = tgt_mask & create_causal_mask(tgt.size(-1)).type_as(tgt_mask.data)   # add causal mask
        return tgt_mask

In [39]:
# 更强大的 batch 格式控制
# more sophisticated batching format
# ref: https://github.com/huggingface/transformers/blob/v4.45.2/src/transformers/tokenization_utils_base.py#L192

from collections import UserDict
class Batch(UserDict):
    """Object for holding a batch of data with mask during training."""

    def __init__(self, src, tgt, pad):
        
        self.data = {}      # ultimate dictionary that stores ('src', 'src_mask', etc.).
        self.data['src'] = src
        self.data['src_mask'] = (src != pad).unsqueeze(-2)
        if tgt is not None:
            # shifted right for next word/token prediction
            self.data['tgt'] = tgt[:, :-1]      # decoder input
            self.data['tgt_y'] = tgt[:, 1:]     # decoder target
            self.data['tgt_mask'] = self.make_std_mask(self.data['tgt'], pad)
            self.data['num_tokens'] = (self.data['tgt_y'] != pad).data.sum()   # num_tokens
        
        super().__init__(self.data)
    
    @staticmethod
    def make_std_mask(tgt, pad):
        "Create a mask to hide padding and future words."
        tgt_mask = (tgt != pad).unsqueeze(-2)
        tgt_mask = tgt_mask & create_causal_mask(tgt.size(-1)).type_as(         # add causal mask
            tgt_mask.data
        )
        return tgt_mask
    
    def __getitem__(self, item: str):
        """
        If the key is a string, returns the value of the dict associated to `key` 
        ('src', 'src_mask', etc.).
        """
        if isinstance(item, str):
            return self.data[item]
        else:
            raise KeyError("Invalid key. Only string of key is available")

    def __getattr__(self, item: str):
        try:
            return self.data[item]
        except KeyError:
            raise AttributeError
        
    def keys(self):
        return self.data.keys()

    def values(self):
        return self.data.values()

    def items(self):
        return self.data.items()

    def device(self):
        return self.data['src'].device

    def to(self, device: Union[str, "torch.device"]):
        """
        Send all values to device by calling `v.to(device)` (PyTorch only).

        Args:
            device (`str` or `torch.device`): The device to put the tensors on.
        """
        
        # This check catches things like APEX blindly calling "to" on all inputs to a module
        # Otherwise it passes the casts down and casts the LongTensor containing the token idxs
        # into a HalfTensor
        if isinstance(device, str) or isinstance(device, int) or isinstance(device, torch.device):
            self.data = {
                k: v.to(device=device) if isinstance(v, torch.Tensor) else v
                for k, v in self.data.items()
            }
        else:
            print(f"warning: Attempting to cast a Batch to type {str(device)}. This is not supported.")
        return self

<br>

## Optimizer and Scheduler

We used the Adam optimizer 
with $\beta_1=0.9$, $\beta_2=0.98$ and $\epsilon=10^{-9}$.  We
varied the learning rate over the course of training, according to
the formula:

$$
lrate = d_{\text{model}}^{-0.5} \cdot
  \min({step\_num}^{-0.5},
    {step\_num} \cdot {warmup\_steps}^{-1.5})
$$

This corresponds to increasing the learning rate linearly for the
first $warmup\_steps$ training steps, and decreasing it thereafter
proportionally to the inverse square root of the step number.  


> We can laso use some other optimizers (e.g., `AdamW`) and learning schedulers (e.g., `CosineAnnealingLR`) implemented in PyTorch or other resources.<br>
> For further details, see: https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
>
> Note: `AdamW` is an improved version of `Adam` and is thus more widely used now.
> <br>https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html#torch.optim.AdamW

In [40]:
def rate(step, model_size, factor, warmup):
    """
    we have to default the step to 1 for LambdaLR function
    to avoid zero raising to negative power.
    """
    if step == 0:
        step = 1
    return factor * (
        model_size ** (-0.5) * min(step ** (-0.5), step * warmup ** (-1.5))
    )


> Example of the curves of this model for different model sizes and
> for optimization hyperparameters.

# three examples: [model_size, factor, warmup]
examples = [
    [256, 1, 200],
    [512, 1, 200],
    [512, 1, 400],
]

dummy_model = torch.nn.Linear(1, 1)

# three examples
results = []
num_steps = 1001
for idx, example in enumerate(examples):
    # run each example
    optimizer = torch.optim.Adam(
        dummy_model.parameters(), lr=1, betas=(0.9, 0.98), eps=1e-9
    )
    lr_scheduler = LambdaLR(
        optimizer=optimizer, lr_lambda=lambda step: rate(step, *example)
    )
    
    # save the learning rate at each step
    for step in range(num_steps):
        lr = optimizer.param_groups[0]["lr"]
        optimizer.step()
        lr_scheduler.step()
        # record each step
        d = {
            'step': step,
            'lr': lr,
            'group': f"{example[0]}:{example[2]}"
        }
        results.append(d)

df = pd.DataFrame(results)

# plot
fig = px.line(df, x="step", y="lr", color="group", title='Learning Rate Schedule', template='none',)
fig.update_layout(
    width=800, height=400,
    xaxis=dict(
        tickmode='linear',
        tick0=0,
        dtick=100,
        range=[0, 1002],
    ),
    yaxis=dict(
        tickmode='linear',
        tick0=0,
        dtick=0.001,
        range=[0, 0.00503],
    ),
)
fig.show()


Image(filename='images/scheduler.png')

<br>

<br>

### Cosine Annealing with Warmup

> The scheduler described above depends heavily on model size ($d_{model}$ or $embed\_dim$) which makes it hard to compare across models.
> <br>In practice, we would like to adopt *Cosine Annealing with Warmup*, a widely used learning rate scheduler nowdays, as the scheduler.
> 
> **Cosine Annealing with Warmup**: 
> <br>This creates a schedule with a learning rate that decreases following the values of the cosine function between the
initial lr set in the optimizer to 0, after a warmup period during which it increases linearly between 0 and the
initial lr set in the optimizer.
> <br>Reference code can be found at: https://github.com/huggingface/transformers/blob/v4.45.2/src/transformers/optimization.py#L144

In [43]:
from torch.optim import Optimizer
from functools import partial

def cosine_schedule_with_warmup_lr_lambda(
    current_step: int, *, num_warmup_steps: int, num_training_steps: int, num_cycles: float = 0.5
):
    if current_step < num_warmup_steps:
        return float(current_step) / float(max(1, num_warmup_steps))
    progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress)))


def cosine_schedule_with_warmup(
    optimizer: Optimizer, num_warmup_steps: int, num_training_steps: int, num_cycles: float = 0.5, last_epoch: int = -1
):
    """
    Create a schedule with a learning rate that decreases following the values of the cosine function between the
    initial lr set in the optimizer to 0, after a warmup period during which it increases linearly between 0 and the
    initial lr set in the optimizer.

    Args:
        optimizer ([`~torch.optim.Optimizer`]):
            The optimizer for which to schedule the learning rate.
        num_warmup_steps (`int`):
            The number of steps for the warmup phase.
        num_training_steps (`int`):
            The total number of training steps.
        num_cycles (`float`, *optional*, defaults to 0.5):
            The number of waves in the cosine schedule (the defaults is to just decrease from the max value to 0
            following a half-cosine).
        last_epoch (`int`, *optional*, defaults to -1):
            The index of the last epoch when resuming training.

    Return:
        `torch.optim.lr_scheduler.LambdaLR` with the appropriate schedule.
    """

    lr_lambda = partial(
        cosine_schedule_with_warmup_lr_lambda,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
        num_cycles=num_cycles,
    )
    return LambdaLR(optimizer, lr_lambda, last_epoch)


> Example of the curves of cosine annealing with warmup scheduler for different number of warmup steps 
> using AdamW as the optimizer

# three examples: [num_warmup_steps, num_training_steps]
examples = [
    [100, 1000],
    [200, 1000],
    [300, 1000],
]

dummy_model = torch.nn.Linear(1, 1)

# three examples
results = []
base_lr = 0.01
num_steps = 1001
for idx, example in enumerate(examples):
    # run each example
    optimizer = torch.optim.AdamW(
        dummy_model.parameters(), lr=base_lr, betas=(0.9, 0.98), eps=1e-9
    )
    # lr_scheduler = LambdaLR(
    #     optimizer=optimizer, lr_lambda=lambda step: rate(step, *example)
    # )

    # lr_scheduler = LambdaLR(
    #     optimizer=optimizer, lr_lambda=lambda step: _get_cosine_schedule_with_warmup_lr_lambda(
    #         step, num_warmup_steps=example[0], num_training_steps=example[1], num_cycles=0.5
    #     )
    # )
    
    lr_scheduler = cosine_schedule_with_warmup(
        optimizer=optimizer, num_warmup_steps=example[0], num_training_steps=example[1]
    )
    
    # save the learning rate at each step
    for step in range(num_steps):
        lr = optimizer.param_groups[0]["lr"]
        optimizer.step()
        lr_scheduler.step()
        # record each step
        d = {
            'step': step,
            'lr': lr,
            'group': f"{example[0]}:{example[1]}"
        }
        results.append(d)

df = pd.DataFrame(results)

# plot
fig = px.line(df, x="step", y="lr", color="group", title='Learning Rate Schedule', template='none',)
fig.update_layout(
    width=800, height=400,
    xaxis=dict(
        tickmode='linear',
        tick0=0,
        dtick=100,
        range=[0, 1002],
    ),
    # yaxis=dict(
    #     tickmode='linear',
    #     tick0=0,
    #     dtick=0.002,
    #     range=[0, 0.0102],
    # ),
)
fig.show()


# image of cosine annealing with warmup scheduler
Image(filename='images/cosine_scheduler.png')

<br>

# Part 3: Toy Training Example: Copy input

> We can begin by trying out a simple copy-task. Given a random set
> of input symbols from a small vocabulary, the goal is to generate
> back those same symbols.



## Synthetic Data

In [ ]:
def data_generator(V, batch_size, nbatches):
    "Generate random data for a src-tgt copy task."
    for i in range(nbatches):
        data = torch.randint(1, V, size=(batch_size, 10))
        data[:, 0] = 1
        src = data.requires_grad_(False).clone().detach()
        tgt = data.requires_grad_(False).clone().detach()
        yield Batch(src, tgt, pad=0)

## Loss Computation

> We simply use cross entropy loss implemented by PyTorch `nn.CrossEntropyLoss`

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)    # cross entropy loss w/ label smoothing

<br>

### Greedy Decoding

> This code predicts a translation using greedy decoding for simplicity.

In [48]:
@torch.inference_mode
def greedy_decode(model, src, src_mask, max_len, start_symbol):
    model.eval()
    memory = model.encode(src, src_mask)
    ys = torch.zeros(1, 1).fill_(start_symbol).type_as(src.data)
    for i in range(max_len - 1):
        causal_mask = create_causal_mask(ys.size(1)).type_as(src.data)
        out = model.decode(ys, memory, src_mask, causal_mask)       # decoder input is placed first
        prob = model.generator(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        ys = torch.cat(
            [ys, torch.zeros(1, 1).type_as(src.data).fill_(next_word)], dim=1
        )
    return ys

<br>

> Next we create a generic training and scoring function to keep
> track of loss. We pass in a generic loss compute function that
> also handles parameter updates.

### Training Loop

In [49]:

class TrainState:
    """Track number of steps, examples, and tokens processed"""
    
    step: int = 0           # Steps in the current epoch
    accum_step: int = 0     # Number of gradient accumulation steps
    samples: int = 0        # total # of examples used
    num_tokens: int = 0     # total # of tokens processed


In [50]:
def run_epoch(
    data_iter,
    model,
    criterion,
    optimizer,
    scheduler,
    mode="train",
    accum_iter=1,
    train_state=TrainState(),
):
    """Train a single epoch"""
    start = time.time()
    total_tokens = 0
    total_loss = 0
    tokens = 0
    n_accum = 0
    for i, batch in enumerate(data_iter):
        batch = batch.to(model.device)      # move inputs to model.device
        logits = model(
            batch.src, batch.tgt, batch.src_mask, batch.tgt_mask
        )
        y_pred = model.generator(logits)
        loss = criterion(y_pred.reshape(-1, y_pred.shape[-1]), batch.tgt_y.reshape(-1))
        if mode == "train" or mode == "train+log":
            loss.backward()
            train_state.step += 1
            train_state.samples += batch.src.shape[0]
            train_state.num_tokens += batch.num_tokens.item()
            if i % accum_iter == 0:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                n_accum += 1
                train_state.accum_step += 1
            scheduler.step()

        total_loss += loss.item()
        total_tokens += batch.num_tokens
        tokens += batch.num_tokens
        if i % 40 == 1 and (mode == "train" or mode == "train+log"):
            lr = optimizer.param_groups[0]["lr"]
            elapsed = time.time() - start
            print(
                (
                    "Epoch Step: %6d | Accumulation Step: %3d | Loss: %6.2f "
                    + "| Tokens / Sec: %7.1f | Learning Rate: %6.1e"
                )
                % (i, n_accum, loss, tokens / elapsed, lr)
            )
            start = time.time()
            tokens = 0
        
    return total_loss / total_tokens, train_state

<br>

## Train the Simple Model

> Note: If the learning rate is set too large, the results may not be good enough.

In [51]:
# Train the simple copy task

def run_toy_example(num_epochs=10, pre_norm=True, device='cpu'):
    V = 11
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)    # cross entropy loss w/ label smoothing
    
    model = create_model(V, V, embed_dim=512, num_layers=2, pre_norm=pre_norm)
    model = model.to(device)
    
    num_batches = 20
    batch_size = 80
    warmup_ratio = 0.1  # 10% of total training steps used for a linear warmup
    num_training_steps = num_epochs * num_batches
    num_warmup_steps = math.ceil(num_training_steps * warmup_ratio)
    base_lr=0.001   # lr is important, large lr heats the performance

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=base_lr, betas=(0.9, 0.98), eps=1e-9
    )
    # lr_scheduler = LambdaLR(
    #     optimizer=optimizer,
    #     lr_lambda=lambda step: rate(
    #         step, model_size=512, factor=1.0, warmup=400
    #     ),
    # )
    lr_scheduler = cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)

    for epoch in range(num_epochs):
        
        model.train()
        run_epoch(
            data_generator(V, batch_size, nbatches=num_batches),
            model,
            criterion,
            optimizer,
            lr_scheduler,
            mode="train",
        )

    
    model.eval()
    # Inference examples
    src = torch.LongTensor([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]]).to(model.device)
    max_len = src.shape[1]
    src_mask = torch.ones(1, 1, max_len).to(model.device)
    pred = greedy_decode(model, src, src_mask, max_len=max_len, start_symbol=0) # set start_symbol as 0
    print(f"\nTarget:    {src}")
    print(f"Predicted: {pred}")

    src = torch.LongTensor([[0, 1, 3, 5, 7, 9]]).to(model.device)
    max_len = src.shape[1]
    src_mask = torch.ones(1, 1, max_len).to(model.device)
    pred = greedy_decode(model, src, src_mask, max_len=max_len, start_symbol=0)
    print(f"\nTarget:    {src}")
    print(f"Predicted: {pred}")
    
    src = torch.LongTensor([[0, 9, 2, 6, 4, 1, 6, 9, 8, 9]]).to(model.device)
    max_len = src.shape[1]
    src_mask = torch.ones(1, 1, max_len).to(model.device)
    pred = greedy_decode(model, src, src_mask, max_len=max_len, start_symbol=0)
    print(f"\nTarget:    {src}")
    print(f"Predicted: {pred}")


# GPU
run_toy_example(num_epochs=20, pre_norm=True, device='cpu')

# GPU
run_toy_example(num_epochs=30, pre_norm=False, device=0)

# CPU
run_toy_example(num_epochs=20, pre_norm=True, device='cpu')